In [4]:
!pip install pandas sentence-transformers faiss-cpu transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 73.0 MB/s eta 0:00:00


In [ ]:

import pandas as pd

data = {
    "question": [
        "What are the college timings?",
        "How can I apply for admission?",
        "What courses are offered?",
        "Is there a hostel facility?",
        "What is the fee structure?",
        "Where is the college located?",
        "How to contact the administration?"
    ],
    "answer": [
        "College runs from 9 AM to 4 PM on weekdays.",
        "You can apply online through the official college website.",
        "We offer B.E, M.Tech, MBA, and diploma courses.",
        "Yes, separate hostel facilities are available for boys and girls.",
        "Fees vary by course. Please check the official website for details.",
        "The college is located in Bangalore, Karnataka.",
        "You can email admin@college.edu or call 1234567890."
    ]
}

df = pd.DataFrame(data)
df.to_csv("college_faqs.csv", index=False)
print("✅ CSV file created successfully!")

# ============================
# STEP 3: Import Libraries
# ============================
import pandas as pd
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# ============================
# STEP 4: Load Dataset
# ============================
df = pd.read_csv("college_faqs.csv")

# Combine Question and Answer for richer context
documents = df["question"] + " " + df["answer"]

print(f"✅ Dataset loaded! Total documents: {len(documents)}")

# ============================
# STEP 5: Create Embeddings
# ============================
print("⏳ Loading embedding model...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

doc_embeddings = embedding_model.encode(documents.tolist())

# Convert to numpy float32 (required by FAISS)
doc_embeddings = np.array(doc_embeddings).astype("float32")

print(f"✅ Embeddings created! Shape: {doc_embeddings.shape}")

# ============================
# STEP 6: Store in FAISS
# ============================
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

print(f"✅ FAISS index created! Vectors indexed: {index.ntotal}")

# ============================
# STEP 7: Load Language Model
# ============================
print("⏳ Loading language model (this may take a minute)...")

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_length=200,
    temperature=0.7
)

print("✅ Language model loaded!")

# ============================
# STEP 8: Retrieval Function
# ============================
def retrieve(query, top_k=2):
    """
    Encodes the query and retrieves the top_k most
    semantically similar documents from FAISS index.
    """
    query_embedding = embedding_model.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    results = []
    for i in indices[0]:
        results.append(documents.iloc[i])
    return results

# ============================
# STEP 9: Generate Answer
# ============================
def generate_answer(query, top_k=2):
    """
    Retrieves relevant context from FAISS and generates
    a human-like answer using the LLM.

    Parameters:
        query  : User's question (string)
        top_k  : Number of documents to retrieve (int, default=2)
    """
    retrieved_docs = retrieve(query, top_k)
    context = " ".join(retrieved_docs)

    prompt = f"""You are a helpful college assistant.
Answer the question based on the context below.

Context: {context}

Question: {query}
Answer:"""

    response = llm(prompt)
    return response[0]['generated_text']

# ============================
# STEP 10: Experimentation
# ============================
# You can experiment with these parameters:

# 1. Change top_k to retrieve more/fewer documents
#    generate_answer("What courses are offered?", top_k=1)
#    generate_answer("What courses are offered?", top_k=3)

# 2. Change temperature in the pipeline (higher = more creative):
#    llm = pipeline("text2text-generation", model="google/flan-t5-base",
#                   max_length=200, temperature=0.1)   # more deterministic
#    llm = pipeline("text2text-generation", model="google/flan-t5-base",
#                   max_length=200, temperature=1.0)   # more creative

# 3. Change the embedding model:
#    embedding_model = SentenceTransformer("all-mpnet-base-v2")  # higher accuracy

# Quick test before chat loop
print("\n--- Quick Test ---")
print("Q: What are the college timings?")
print("A:", generate_answer("What are the college timings?"))
print("------------------\n")

# ============================
# STEP 11: Interactive Chat Loop
# ============================
print("🎓 College FAQ Chatbot (type 'exit' to quit)\n")

while True:
    user_input = input("You: ")

    if user_input.strip() == "":
        continue

    if user_input.lower() in ["exit", "quit", "bye"]:
        print("Chatbot: Goodbye! Have a great day! 👋")
        break

    answer = generate_answer(user_input, top_k=2)
    print("Chatbot:", answer)
    print()

✅ CSV file created successfully!
✅ Dataset loaded! Total documents: 7
⏳ Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_length', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 

✅ Language model loaded!

--- Quick Test ---
Q: What are the college timings?
A: You are a helpful college assistant.
Answer the question based on the context below.

Context: What are the college timings? College runs from 9 AM to 4 PM on weekdays. How can I apply for admission? You can apply online through the official college website.

Question: What are the college timings?
Answer:
------------------

🎓 College FAQ Chatbot (type 'exit' to quit)

You: Does your college provide hostel?
Chatbot: You are a helpful college assistant.
Answer the question based on the context below.

Context: Is there a hostel facility? Yes, separate hostel facilities are available for boys and girls. Where is the college located? The college is located in Bangalore, Karnataka.

Question: Does your college provide hostel?
Answer:

You: Does your college provide hostel?
Chatbot: You are a helpful college assistant.
Answer the question based on the context below.

Context: Is there a hostel facility? Yes, s